In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
for k in ['KAGGLE_USERNAME', 'KAGGLE_KEY', 'HF_TOKEN', 'OPENAI_API_KEY']:
    os.environ[k] = userdata.get(k)
os.environ['HUGGINGFACE_TOKEN'] = os.environ['HF_TOKEN']

Mounted at /content/drive


In [ ]:
!git clone https://github.com/benyadabest/contrastive_learning_clinical_embeddings.git
%cd contrastive_learning_clinical_embeddings
!pip install -q -r requirements.txt kaggle

Cloning into 'contrastive_learning_clinical_embeddings'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 77 (delta 32), reused 60 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 1.57 MiB | 20.32 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/contrastive_learning_clinical_embeddings


In [ ]:
CACHE = '/content/drive/MyDrive/medical_embeddings/mimic_iii_10k_cache'
TARGET = 'MIMIC -III (10000 patients)'

import shutil, os, zipfile
from pathlib import Path
"""
os.makedirs(CACHE, exist_ok=True)

if not os.path.exists(f'{CACHE}/{TARGET}'):
    !kaggle datasets download -d bilal1907/mimic-iii-10k -p /content
    !unzip -q /content/mimic-iii-10k.zip -d /content/mimic_extracted
    !ls /content/mimic_extracted   # <-- inspect actual structure
    # then move into Drive cache (see verification step below)
else:
    shutil.copytree(f'{CACHE}/{TARGET}', TARGET)
"""

os.makedirs(CACHE, exist_ok=True)

if not os.path.exists(f'{CACHE}/{TARGET}'):
    !kaggle datasets download -d bilal1907/mimic-iii-10k -p /content

    zip_path = "/content/mimic-iii-10k.zip"
    extract_root = "/content/mimic_extracted"

    os.makedirs(extract_root, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as zf:
        all_files = zf.namelist()

        dir_to_files = {}

        for f in all_files:
            if f.endswith('/'):
                continue

            parent = str(Path(f).parent)

            dir_to_files.setdefault(parent, []).append(f)

        files_to_extract = []

        for parent, files in dir_to_files.items():

            csv_files = [f for f in files if f.endswith('.csv')]

            if len(csv_files) == 1:
                files_to_extract.extend(csv_files)

            else:
                sorted_files = [
                    f for f in csv_files
                    if f.endswith('_sorted.csv')
                ]

                files_to_extract.extend(sorted_files)

        print(f"Extracting {len(files_to_extract)} files...")

        for f in files_to_extract:
            zf.extract(f, extract_root)

    print("Done.")

else:
    shutil.copytree(f'{CACHE}/{TARGET}', TARGET)

In [ ]:
MIMIC_DIR = 'MIMIC -III (10000 patients)'
!python src/preprocess.py --mimic-dir "{MIMIC_DIR}"

Loading data...
ICD hierarchy: 12911 admissions with diagnoses
Admissions summary: 12911 rows
Diagnosis labels: 118300 rows
Temporal pairs: 470471 pairs from 10000 patients
Note-level dataset: 480471 notes

Outputs saved to: /content/contrastive_learning_clinical_embeddings/data


In [ ]:
import shutil, os
SRC = '/content/mimic_extracted/MIMIC -III (10000 patients)'
DST = '/content/drive/MyDrive/medical_embeddings/mimic_iii_10k_cache/MIMIC -III (10000 patients)'
if not os.path.exists(DST):
    print("Copying to Drive (~1.2 GB, takes a few min)...")
    shutil.copytree(SRC, DST)
    print("Cached.")
else:
    print("Already cached.")

Already cached.


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA build:", torch.version.cuda)
print("Torch version:", torch.__version__)

CUDA available: True
Torch CUDA build: 12.8
Torch version: 2.10.0+cu128


In [ ]:
!python src/dataset_reduce.py

(633, 22)
(23657, 9)
(24141, 12)


In [ ]:
EMB_SRC = '/content/contrastive_learning_clinical_embeddings/embeddings'
EMB_DST = '/content/drive/MyDrive/medical_embeddings/embeddings_cache'
PAIRS = 'data/temporal_pairs_small.json'
NOTES = 'data/notes_with_icd_small.csv'

if os.path.exists(EMB_DST):
  print("Copying embeddings from Drive")
  shutil.copytree(EMB_DST, EMB_SRC)
else:
  !python src/embed.py --mode pairs --input "{PAIRS}" --model google/embeddinggemma-300m
  !python src/embed.py --mode pairs --input "{PAIRS}" --model text-embedding-3-small
  !python src/embed.py --mode pairs --input "{PAIRS}" --model text-embedding-3-large

Copying embeddings from Drive


In [ ]:
if not os.path.exists(EMB_DST):
    print("Copying embeddings to Drive")
    shutil.copytree(EMB_SRC, EMB_DST)
    print("Cached.")
else:
    print("Already cached.")

Already cached.


In [ ]:
!python src/embed.py --mode notes --input "{NOTES}" --model google/embeddinggemma-300m
!python src/embed.py --mode notes --input "{NOTES}" --model text-embedding-3-small
!python src/embed.py --mode notes --input "{NOTES}" --model text-embedding-3-large

In [ ]:
PAIRS = 'data/temporal_pairs_small.json'

GEMMA_INFONCE_BEST_SRC = '/content/contrastive_learning_clinical_embeddings/models/embeddinggemma_infonce_best'
GEMMA_INFONCE_BEST_DST = '/content/drive/MyDrive/medical_embeddings/embeddinggemma_infonce_best'

GEMMA_HIERARCHICAL_BEST_SRC = '/content/contrastive_learning_clinical_embeddings/models/embeddinggemma_hierarchical_best'
GEMMA_HIERARCHICAL_BEST_DST = '/content/drive/MyDrive/medical_embeddings/embeddinggemma_hierarchical_best'

import gc
gc.collect()
torch.cuda.empty_cache()

# InfoNCE - temporal contrastive baseline
if os.path.exists(GEMMA_INFONCE_BEST_DST):
  if not os.path.exists(GEMMA_INFONCE_BEST_SRC):
    print("Copying cached best gemma model with infoNCE loss to models/")
    shutil.copytree(GEMMA_INFONCE_BEST_DST, GEMMA_INFONCE_BEST_SRC)
else:
  !python src/train_contrastive.py --pairs "{PAIRS}" --loss infonce --epochs 5

# Hierarchical - your extension
if os.path.exists(GEMMA_HIERARCHICAL_BEST_DST):
  if not os.path.exists(GEMMA_HIERARCHICAL_BEST_SRC):
    print("Copying cached best gemma model with hierarchical loss to models/")
    shutil.copytree(GEMMA_HIERARCHICAL_BEST_DST, GEMMA_HIERARCHICAL_BEST_SRC)
else:
  !python src/train_contrastive.py --pairs "{PAIRS}" --loss hierarchical --epochs 10

Loading model: google/embeddinggemma-300m
Loading weights: 100% 314/314 [00:00<00:00, 1071.26it/s, Materializing param=norm.weight]
Device: cuda
Dataset: 23657 pairs, 739 batches
Epoch 1/10: 100% 739/739 [23:01<00:00,  1.87s/it, loss=1.3142]
Epoch 1: avg_loss = 1.3342
Writing model shards: 100% 1/1 [00:02<00:00,  2.10s/it]
  Saved best model to /content/contrastive_learning_clinical_embeddings/models/embeddinggemma_hierarchical_best
Epoch 2/10: 100% 739/739 [23:02<00:00,  1.87s/it, loss=0.3755]
Epoch 2: avg_loss = 0.7082
Writing model shards: 100% 1/1 [00:02<00:00,  2.70s/it]
  Saved best model to /content/contrastive_learning_clinical_embeddings/models/embeddinggemma_hierarchical_best
Epoch 3/10: 100% 739/739 [23:02<00:00,  1.87s/it, loss=0.2662]
Epoch 3: avg_loss = 0.3767
Writing model shards: 100% 1/1 [00:02<00:00,  2.70s/it]
  Saved best model to /content/contrastive_learning_clinical_embeddings/models/embeddinggemma_hierarchical_best
Epoch 4/10: 100% 739/739 [23:01<00:00,  1.87s/i

In [ ]:
if not os.path.exists(GEMMA_INFONCE_BEST_DST):
    print("Copying best gemma model with infoNCE loss to Drive")
    shutil.copytree(GEMMA_INFONCE_BEST_SRC, GEMMA_INFONCE_BEST_DST)
    print("Cached.")
else:
    print("Already cached.")

if not os.path.exists(GEMMA_HIERARCHICAL_BEST_DST):
    print("Copying best gemma model with hierarchical loss to Drive")
    shutil.copytree(GEMMA_HIERARCHICAL_BEST_SRC, GEMMA_HIERARCHICAL_BEST_DST)
    print("Cached.")
else:
    print("Already cached.")

Already cached.
Copying best gemma model with hierarchical loss to Drive
Cached.


In [ ]:
GEMMA_INFONCE = 'models_embeddinggemma_infonce_best'
GEMMA_HIERARCHICAL = 'models_embeddinggemma_hierarchical_best'

if not os.path.exists(os.path.join(EMB_SRC, f'anchor_embeddings_{GEMMA_INFONCE}.npy')) and \
  os.path.exists(os.path.join(EMB_SRC, f'positive_embeddings_{GEMMA_INFONCE}.npy')):
  !python src/embed.py --mode pairs --input data/temporal_pairs_small.json --model models/embeddinggemma_infonce_best

if not os.path.exists(os.path.join(EMB_SRC, f'anchor_embeddings_{GEMMA_HIERARCHICAL}.npy')) and \
  os.path.exists(os.path.join(EMB_SRC, f'positive_embeddings_{GEMMA_HIERARCHICAL}.npy')):
  !python src/embed.py --mode pairs --input data/temporal_pairs_small.json --model models/embeddinggemma_hierarchical_best

In [ ]:
if not os.path.exists(os.path.join(EMB_DST, f'anchor_embeddings_{GEMMA_INFONCE}.npy')) and \
  os.path.exists(os.path.join(EMB_DST, f'positive_embeddings_{GEMMA_INFONCE}.npy')):
  shutil.copy(f'embeddings/anchor_embeddings_{GEMMA_INFONCE}.npy', EMB_DST)
  shutil.copy(f'embeddings/positive_embeddings_{GEMMA_INFONCE}', EMB_DST)

if not os.path.exists(os.path.join(EMB_DST, f'anchor_embeddings_{GEMMA_HIERARCHICAL}.npy')) and \
  os.path.exists(os.path.join(EMB_DST, f'positive_embeddings_{GEMMA_HIERARCHICAL}.npy')):
  shutil.copy(f'embeddings/anchor_embeddings_{GEMMA_HIERARCHICAL}.npy', EMB_DST)
  shutil.copy(f'embeddings/positive_embeddings_{GEMMA_HIERARCHICAL}.npy', EMB_DST)

In [ ]:
!python src/embed.py --mode notes --input data/notes_with_icd_small.csv --model models/embeddinggemma_infonce_best
!python src/embed.py --mode notes --input data/notes_with_icd_small.csv --model models/embeddinggemma_hierarchical_best

Loaded 24141 texts from data/notes_with_icd_small.csv
Loading weights: 100% 314/314 [00:00<00:00, 1205.36it/s, Materializing param=norm.weight]
Batches: 100% 755/755 [03:39<00:00,  3.44it/s]
Saved embeddings: (24141, 768) -> /content/contrastive_learning_clinical_embeddings/embeddings/embeddings_models_embeddinggemma_infonce_best.npy
Loaded 24141 texts from data/notes_with_icd_small.csv
Loading weights: 100% 314/314 [00:00<00:00, 936.72it/s, Materializing param=norm.weight] 
Batches: 100% 755/755 [03:39<00:00,  3.43it/s]
Saved embeddings: (24141, 768) -> /content/contrastive_learning_clinical_embeddings/embeddings/embeddings_models_embeddinggemma_hierarchical_best.npy


In [ ]:
!python src/evaluate.py --task compare --notes data/notes_with_icd_small.csv


Evaluating: text-embedding-3-small

Note Recall:
  Top-1 recall accuracy: 0.0031 (74/23657)
  Top-5 recall accuracy: 0.0514 (1216/23657)
  Top-10 recall accuracy: 0.0644 (1524/23657)

Diagnosis Prediction:
  Using top 25 ICD codes (most frequent)
  Train: 16588, Test: 4147
  Macro AUROC: 0.8949

Evaluating: text-embedding-3-large

Note Recall:
  Top-1 recall accuracy: 0.0031 (74/23657)
  Top-5 recall accuracy: 0.0545 (1289/23657)
  Top-10 recall accuracy: 0.0699 (1654/23657)

Diagnosis Prediction:
  Using top 25 ICD codes (most frequent)
  Train: 16588, Test: 4147
  Macro AUROC: 0.9049

Evaluating: google/embeddinggemma-300m

Note Recall:
  Top-1 recall accuracy: 0.0035 (82/23657)
  Top-5 recall accuracy: 0.0599 (1417/23657)
  Top-10 recall accuracy: 0.0757 (1792/23657)

Diagnosis Prediction:
  Using top 25 ICD codes (most frequent)
  Train: 16588, Test: 4147
  Macro AUROC: 0.8974

Evaluating: models_embeddinggemma_infonce_best

Note Recall:
  Top-1 recall accuracy: 0.0084 (198/23657)

In [ ]:
!python src/evaluate.py --task umap --notes data/notes_with_icd_small.csv \
    --embeddings embeddings/anchor_embeddings_models_embeddinggemma_hierarchical_best.npy \
    --output-name umap_embeddings_hierarchical.png

!python src/evaluate.py --task umap --notes data/notes_with_icd_small.csv \
    --embeddings embeddings/anchor_embeddings_models_embeddinggemma_infonce_best.npy \
    --output-name umap_embeddings_infonce.png

2026-05-14 09:16:22.942130: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-14 09:16:23.013679: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
  Running UMAP...
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
  UMAP saved to /content/contrastive_learning_clinical_embeddings/results/umap_embeddings_hierarchical.png
2026-05-14 09:17:12.812347: I tensorflow/core/util/port.cc:153] oneDNN custom opera

In [ ]:
shutil.copytree('/content/contrastive_learning_clinical_embeddings/results', '/content/drive/MyDrive/medical_embeddings/medical_embeddings_results')





'/content/drive/MyDrive/medical_embeddings_results'